In [1]:
import feedparser
import pandas as pd
from datetime import date, timedelta
from sqlalchemy import create_engine

engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()

clean_path = "../data/clean/"
raw_path = "../data/raw/"
csv_path = "\\Users\\User\\iCloudDrive\\"
one_path = "\\Users\\User\\OneDrive\\Documents\\Data\\"
pdf_path = "../PDF/"

pd.set_option('display.max_colwidth', 255)
pd.set_option('display.max_rows',None)
url = "https://feeds.feedburner.com/Setorth-form45-en"

today = date.today()
year = 2022
mmdd_str = today.strftime('%m%d')
mmdd_str

'0301'

In [2]:
#today = date(2023, 2, 24)
mmdd_str = today.strftime('%m%d')
mmdd_str

'0301'

In [3]:
rss_source = feedparser.parse(url)
f45_number = len(rss_source.entries)
print("Number of F45: ", f45_number)

Number of F45:  32


In [4]:
f45_items = []

for x in range(f45_number):
    f45_content = rss_source.entries[x]
    f45_item = {}
    
    print("\n----------------------------------\n")
    
    print("F45: " + str(x))
    title_ary = f45_content.title.partition(' ')
    f45_item['name'] = title_ary[0].strip() 
    print("Title: ", f45_item['name'])  
    f45_item['year'] = year
    print("Year: ", f45_item['year'])      
    qtr_ary = title_ary[2].partition(' (F45)')
    f45_item['quarter'] = qtr_ary[0][-1]    
    print("Quarter: ", f45_item['quarter'])    
    f45_item['link'] = f45_content.link
    print("Link: ", f45_item['link'])
    f45_item['published'] = f45_content.published
    print("Published: ", f45_item['published'])  
    f45_items.append(f45_item)


----------------------------------

F45: 0
Title:  TRUE
Year:  2022
Quarter:  y
Link:  https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256351610&symbol=TRUE
Published:  Wed, 01 Mar 2023 17:54:31 +0700

----------------------------------

F45: 1
Title:  TH
Year:  2022
Quarter:  y
Link:  https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256325100&symbol=TH
Published:  Wed, 01 Mar 2023 12:36:22 +0700

----------------------------------

F45: 2
Title:  PK
Year:  2022
Quarter:  y
Link:  https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256324800&symbol=PK
Published:  Wed, 01 Mar 2023 12:30:42 +0700

----------------------------------

F45: 3
Title:  APCS
Year:  2022
Quarter:  y
Link:  https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256317420&symbol=APCS
Published:  Wed, 01 Mar 2023 08:55:48 +0700

----------------------------------

F45: 4
Title:  TCOAT
Year:  2022
Quarter:  y
Link:  https://www.set.or.th/en/market/new

In [5]:
df = pd.DataFrame(f45_items)
df[['name','year','quarter','published']]

,name,year,quarter,published
0,TRUE,2022,y,"Wed, 01 Mar 2023 17:54:31 +0700"
1,TH,2022,y,"Wed, 01 Mar 2023 12:36:22 +0700"
2,PK,2022,y,"Wed, 01 Mar 2023 12:30:42 +0700"
3,APCS,2022,y,"Wed, 01 Mar 2023 08:55:48 +0700"
4,TCOAT,2022,y,"Wed, 01 Mar 2023 08:45:12 +0700"
5,TH,2022,y,"Wed, 01 Mar 2023 08:43:53 +0700"
6,IFEC,2022,y,"Wed, 01 Mar 2023 08:34:29 +0700"
7,JCK,2022,y,"Wed, 01 Mar 2023 08:28:13 +0700"
8,CRANE,2022,y,"Wed, 01 Mar 2023 08:27:58 +0700"
9,THL,2022,y,"Wed, 01 Mar 2023 08:27:40 +0700"


In [6]:
df.groupby(['year','quarter']).count()

,,name,link,published
year,quarter,,,
2022,y,32,32,32


In [7]:
df.loc[(df.quarter == 'y') ,['year','quarter']] = ['2022','4']
df.groupby(['year','quarter']).count()

,,name,link,published
year,quarter,,,
2022,4,32,32,32


In [8]:
df.loc[(df.quarter == 'e') ,['year','quarter']] = ['2022','4']
df.groupby(['year','quarter']).count()

,,name,link,published
year,quarter,,,
2022,4,32,32,32


In [9]:
df.loc[(df.quarter == 'S') ,['year','quarter']] = ['2022','4']
df.groupby(['year','quarter']).count()

,,name,link,published
year,quarter,,,
2022,4,32,32,32


In [10]:
df.loc[(df.quarter == '2') ,['year','quarter']] = ['2022','4']
df.groupby(['year','quarter']).count()

,,name,link,published
year,quarter,,,
2022,4,32,32,32


In [11]:
df.loc[(df.quarter == '2') ,['year','qAuarter']] = ['2022','4']
df.groupby(['year','quarter']).count()

name  link  published  qAuarter
year quarter                                 
2022 2           1     1          1         1
     4          68    68         68         0

In [12]:
df.loc[(df.name == 'GVREIT') ,['year','quarter']] = ['2023','1']
df.groupby(['year','quarter']).count()

name  link  published  qAuarter
year quarter                                 
2022 2           1     1          1         1
     4          68    68         68         0

### After change all illogical quarters

In [11]:
df.quarter = df.quarter.astype(int)
df.groupby(['year','quarter']).count()

,,name,link,published
year,quarter,,,
2022,4,32,32,32


In [12]:
### First equals to latest published pdf file
df = df.drop_duplicates(subset='name', keep='first')
df.shape

(28, 5)

In [13]:
file_name = 'F45-RAW-' + mmdd_str + '.csv'
raw_file = raw_path + file_name
output_file = csv_path + file_name
one_file = one_path + file_name

df[['name','year','quarter','published']].to_csv(output_file, header=True, index=False, sep=',')
df[['name','year','quarter','published']].to_csv(one_file,    header=True, index=False, sep=',')
df[['name','year','quarter','published']].to_csv(raw_file,    header=True, index=False, sep=',')

In [14]:
sql = '''
SELECT *
FROM exempts
ORDER BY name'''
df_exempts = pd.read_sql(sql, conlt)
df_exempts.shape[0]

403

In [15]:
df_merge = pd.merge(df, df_exempts, on='name', how='outer', indicator=True)
df_merge.shape

(417, 7)

### Tickers that are in Exempts table

In [16]:
in_exempts = df_merge.loc[
    df_merge['_merge'] == 'both',
    ['name','year','quarter','published','link']
    
]
in_exempts.year = in_exempts.year.astype(int)
in_exempts.quarter = in_exempts.quarter.astype(int)
in_exempts.sort_values(by=['published'],ascending=[False])

,name,year,quarter,published,link
1,TH,2022,4,"Wed, 01 Mar 2023 12:36:22 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256325100&symbol=TH
3,APCS,2022,4,"Wed, 01 Mar 2023 08:55:48 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256317420&symbol=APCS
4,TCOAT,2022,4,"Wed, 01 Mar 2023 08:45:12 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256316190&symbol=TCOAT
7,CRANE,2022,4,"Wed, 01 Mar 2023 08:27:58 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256312990&symbol=CRANE
10,JKN,2022,4,"Wed, 01 Mar 2023 08:19:06 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256311350&symbol=JKN
12,KOOL,2022,4,"Wed, 01 Mar 2023 08:08:38 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256309790&symbol=KOOL
13,HUMAN,2022,4,"Wed, 01 Mar 2023 08:08:27 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256309760&symbol=HUMAN
14,TRITN,2022,4,"Wed, 01 Mar 2023 08:07:26 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256309570&symbol=TRITN
15,WAVE,2022,4,"Wed, 01 Mar 2023 08:07:06 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256309440&symbol=WAVE
16,BR,2022,4,"Wed, 01 Mar 2023 08:03:02 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256308900&symbol=BR


In [17]:
in_exempts.sort_values(by=['published'],ascending=[False]).shape[0]

14

### Not in exempts table

In [18]:
df_out = df_merge.loc[
    df_merge['_merge'] == 'left_only',
    ['name','year','quarter','published','link']
]
df_out.year = df_out.year.astype(int)
df_out.quarter = df_out.quarter.astype(int)
df_out.sort_values(by=['published'],ascending=[False])

,name,year,quarter,published,link
0,TRUE,2022,4,"Wed, 01 Mar 2023 17:54:31 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256351610&symbol=TRUE
2,PK,2022,4,"Wed, 01 Mar 2023 12:30:42 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256324800&symbol=PK
5,IFEC,2022,4,"Wed, 01 Mar 2023 08:34:29 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256314280&symbol=IFEC
6,JCK,2022,4,"Wed, 01 Mar 2023 08:28:13 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256313010&symbol=JCK
8,THL,2022,4,"Wed, 01 Mar 2023 08:27:40 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256312950&symbol=THL
9,PRO,2022,4,"Wed, 01 Mar 2023 08:23:15 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256311940&symbol=PRO
11,STECH,2022,4,"Wed, 01 Mar 2023 08:17:18 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256311200&symbol=STECH
17,GLOCON,2022,4,"Wed, 01 Mar 2023 08:02:31 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256308780&symbol=GLOCON
19,DIMET,2022,4,"Wed, 01 Mar 2023 07:38:18 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256306910&symbol=DIMET
20,BYD,2022,4,"Wed, 01 Mar 2023 07:31:19 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256306600&symbol=BYD


In [19]:
#df_out = df_out.drop(df_out.index[df_out['name'] == "SCC"])
df_out.shape[0]

14

In [20]:
sql = '''
SELECT *
FROM tickers
ORDER BY name'''
df_tickers = pd.read_sql(sql, conlt)
df_tickers.shape

(401, 9)

In [21]:
df_merge2 = pd.merge(df_out, df_tickers, on='name', how='outer', indicator=True)
df_merge2.shape

(413, 14)

### There are no ticker records

In [22]:
df_merge2.loc[
    df_merge2['_merge'] == 'left_only',
    ['name','year','quarter','published','link']
]

,name,year,quarter,published,link
1,PK,2022.0,4.0,"Wed, 01 Mar 2023 12:30:42 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256324800&symbol=PK
2,IFEC,2022.0,4.0,"Wed, 01 Mar 2023 08:34:29 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256314280&symbol=IFEC
3,JCK,2022.0,4.0,"Wed, 01 Mar 2023 08:28:13 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256313010&symbol=JCK
4,THL,2022.0,4.0,"Wed, 01 Mar 2023 08:27:40 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256312950&symbol=THL
5,PRO,2022.0,4.0,"Wed, 01 Mar 2023 08:23:15 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256311940&symbol=PRO
6,STECH,2022.0,4.0,"Wed, 01 Mar 2023 08:17:18 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256311200&symbol=STECH
7,GLOCON,2022.0,4.0,"Wed, 01 Mar 2023 08:02:31 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256308780&symbol=GLOCON
8,DIMET,2022.0,4.0,"Wed, 01 Mar 2023 07:38:18 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256306910&symbol=DIMET
9,BYD,2022.0,4.0,"Wed, 01 Mar 2023 07:31:19 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256306600&symbol=BYD
10,JCKH,2022.0,4.0,"Wed, 01 Mar 2023 07:28:54 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256306430&symbol=JCKH


In [23]:
df_merge2.loc[
    df_merge2['_merge'] == 'left_only',
    ['name','year','quarter','published','link','id','market']
].shape

(12, 7)

### There are ticker records

In [24]:
df_out2 = df_merge2.loc[
    df_merge2['_merge'] == 'both',
    ['name','year','quarter','published','link','id','market']
]
df_out2

,name,year,quarter,published,link,id,market
0,TRUE,2022.0,4.0,"Wed, 01 Mar 2023 17:54:31 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256351610&symbol=TRUE,571.0,SET50 / SETTHSI
11,BCH,2022.0,4.0,"Wed, 01 Mar 2023 06:59:50 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256304510&symbol=BCH,51.0,SET100 / SETWB


In [25]:
df_out2 = df_out2[df_out2.year.notnull()]
df_out2.shape

(2, 7)

In [26]:
df_out2['year'] = df_out2['year'].astype(int)
df_out2['quarter'] = df_out2['quarter'].astype(int)
df_out2.shape

(2, 7)

In [27]:
file_name = 'F45-CLEAN-' + mmdd_str + '.csv'
clean_file = clean_path + file_name
output_file = csv_path + file_name
one_file = one_path + file_name

df_out2[['name','year','quarter','published','market']].sort_values(['published'],ascending=[False]).to_csv(output_file, header=True, index=False, sep=',')
df_out2[['name','year','quarter','published','market']].sort_values(['published'],ascending=[False]).to_csv(clean_file, header=True, index=False, sep=',')
df_out2[['name','year','quarter','published','market']].sort_values(['published'],ascending=[False]).to_csv(one_file, header=True, index=False, sep=',')

In [28]:
sql = '''
SELECT * 
FROM epss
WHERE year = 2022'''
df_epss = pd.read_sql(sql, conlt)
df_epss.shape

(906, 14)

In [29]:
df_merge3 = pd.merge(df_out2, df_epss, on=['name','year','quarter'], how='outer', indicator=True)
df_merge3.shape

(908, 19)

### Already input, display profit amt & eps to check with new F45

In [30]:
df_merge3[df_merge3['_merge'] == 'both']

,name,year,quarter,published,link,id_x,market,id_y,q_amt,y_amt,aq_amt,ay_amt,q_eps,y_eps,aq_eps,ay_eps,ticker_id,publish_date,_merge


In [31]:
df_merge3[df_merge3['_merge'] == 'both'].shape

(0, 19)

### New F-45

In [32]:
df_inp2 = df_merge3[df_merge3['_merge'] == 'left_only']
df_inp3 = df_inp2.copy()
df_inp3.shape

(2, 19)

In [33]:
df_inp3['year'] = df_inp3['year'].astype(str)
df_inp3['quarter'] = df_inp3['quarter'].astype(str)
final = df_inp3.sort_values('name',ascending=True)
final_str = final.name+' '+final.year+' '+final.quarter+' '+final.link
final_str

1      BCH 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256304510&symbol=BCH
0    TRUE 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256351610&symbol=TRUE
dtype: object

In [34]:
df_inp3_q3 = df_inp3[df_inp3['quarter'] == '4']
final_q3 = df_inp3_q3.sort_values('name',ascending=True)
final_str_q3 = final_q3.name+' '+final_q3.year+' '+final_q3.quarter+' '+final_q3.link
final_str_q3

1      BCH 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256304510&symbol=BCH
0    TRUE 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16776256351610&symbol=TRUE
dtype: object

In [35]:
df_inp3_q1 = df_inp3[df_inp3['quarter'] != '4']
final_q1 = df_inp3_q1.sort_values('name',ascending=True)
final_str_q1 = final_q1.name+' '+final_q1.year+' '+final_q1.quarter+' '+final_q1.link
final_str_q1

Series([], dtype: object)